# درس 12: جبر خطی، ماتریس کوواریانس و PCA

متن کامل درس: [https://quantacademy.ir/courses/quant-math/linear-algebra-covariance-pca/](https://quantacademy.ir/courses/quant-math/linear-algebra-covariance-pca/)

داده‌های این دوره **شبیه‌سازی‌شده** هستند، نه قیمت واقعی بازار؛ هدف یاد دادن روش است. آموزشی است و توصیه سرمایه‌گذاری نیست.

سلول‌ها را از بالا به پایین اجرا کن (Shift+Enter).

## ۱. ماتریس بازده و بردار وزن

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(12)
n_days, n_assets = 1000, 8
market = rng.normal(0, 0.02, n_days)
sector = rng.normal(0, 0.01, n_days)
beta = np.array([1.0, 1.1, 0.9, 1.2, 0.8, 1.0, 1.1, 0.9])
gamma = np.array([1, 1, 1, 1, -1, -1, -1, -1], dtype=float)
noise = rng.normal(0, 0.012, (n_days, n_assets))
R = np.outer(market, beta) + np.outer(sector, gamma) + noise

w = np.full(n_assets, 1 / n_assets)
port = R @ w
print(R.shape)
print(round(port.mean(), 5), round(port.std(ddof=1), 5))

## ۲. ماتریس کوواریانس

In [ ]:
S = np.cov(R, rowvar=False)
print(np.allclose(S, S.T))
print(round(np.sqrt(w @ S @ w), 5), round(port.std(ddof=1), 5))
print(np.round(np.sqrt(np.diag(S)), 4))

## ۳. مقدار و بردار ویژه

In [ ]:
vals, vecs = np.linalg.eigh(S)
order = np.argsort(vals)[::-1]
vals, vecs = vals[order], vecs[:, order]
print(vals.round(6))
print((vals / vals.sum()).round(3))
print(bool((vals >= 0).all()), round(vals.sum(), 6), round(np.trace(S), 6))

## ۴. PCA: پیدا کردن عامل‌ها از داده

In [ ]:
pc1 = vecs[:, 0] * np.sign(vecs[:, 0].sum())
pc2 = vecs[:, 1] * np.sign(vecs[0, 1])
print(pc1.round(2))
print(pc2.round(2))

scores = (R - R.mean(axis=0)) @ vecs
print(round(abs(np.corrcoef(scores[:, 0], market)[0, 1]), 3),
      round(abs(np.corrcoef(scores[:, 1], sector)[0, 1]), 3))

## ۵. ماتریس کوواریانس نمونه، پرنویز است

In [ ]:
def spectrum(n, T, seed=0):
    X = np.random.default_rng(seed).normal(size=(T, n))
    ev = np.linalg.eigvalsh(np.cov(X, rowvar=False))
    return ev.min(), ev.max(), ev.max() / ev.min()

print("T      min    max    cond   MP_min MP_max")
for T in (60, 250, 1000, 10000):
    lo, hi, cond = spectrum(50, T)
    mp_lo, mp_hi = (1 - np.sqrt(50 / T)) ** 2, (1 + np.sqrt(50 / T)) ** 2
    print(f"{T:<6} {lo:<6.3f} {hi:<6.3f} {cond:<6.1f} {mp_lo:<6.3f} {mp_hi:.3f}")

In [ ]:
print(round(np.linalg.cond(S), 1))